# Notebook 02：数据清洗与收益率

目标：检查 OHLCV 数据质量，计算日收益率和累计收益。

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DB_PATH = ROOT / 'data' / 'stocks.db'
print(f'项目根目录: {ROOT}')
print(f'数据库: {DB_PATH}')

In [ ]:
import pandas as pd
import numpy as np
from src.database import get_connection, get_stock_data

conn = get_connection(str(DB_PATH))
code_id = '000001'
df = get_stock_data(conn, code_id)
conn.close()

df = df.sort_values('date').reset_index(drop=True)
print(df.dtypes)
df.head()

In [ ]:
quality = pd.DataFrame({
    '缺失值': df.isna().sum(),
    '零值数量': (df[['open', 'high', 'low', 'close', 'volume']] == 0).sum().reindex(df.columns, fill_value=0),
})
quality

In [ ]:
clean = df.copy()
price_cols = ['open', 'high', 'low', 'close']
clean[price_cols] = clean[price_cols].replace(0, np.nan).ffill()
clean['return'] = clean['close'].pct_change()
clean['log_return'] = np.log(clean['close'] / clean['close'].shift(1))
clean['cum_return'] = (1 + clean['return'].fillna(0)).cumprod() - 1
clean[['date', 'close', 'return', 'log_return', 'cum_return']].tail()

In [ ]:
summary = clean[['return', 'log_return', 'cum_return']].describe().T
summary

小结：清洗逻辑保持简单透明，正式平台在读取数据时保持原始 OHLCV，再由分析和回测模块按需计算指标。